In [ ]:
print("kernel alive")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vetuner.drive_cycle import generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table

cycle = generate_drive_cycle()

base_table = ve_table(params=BASE_MAP_SURFACE)
result = simulate_engine(cycle, base_table)

fuelled = ~result.fuel_cut
print(f"{len(result)} samples, {result.fuel_cut.sum()} in fuel cut")
print(f"AFR range while fuelled: {np.nanmin(result.afr_actual):.2f} "
      f"to {np.nanmax(result.afr_actual):.2f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(result.time_s, result.rpm, lw=1.0, color="tab:blue")
axes[0].set_ylabel("engine speed (rpm)")

axes[1].plot(result.time_s, result.map_kpa, lw=1.0, color="tab:green")
axes[1].set_ylabel("MAP (kPa abs)")

axes[2].plot(result.time_s, result.afr_target, lw=1.2, color="black",
             ls="--", label="target")
axes[2].plot(result.time_s, result.afr_actual, lw=1.0, color="tab:red",
             label="actual")
axes[2].set_ylabel("air-fuel ratio")
axes[2].set_xlabel("time (s)")
axes[2].legend(loc="upper right")

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("Engine response with a miscalibrated VE table")
fig.tight_layout()

In [ ]:
error_ratio = result.afr_actual / result.afr_target

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    result.rpm[fuelled],
    error_ratio[fuelled],
    c=result.map_kpa[fuelled],
    s=6,
    alpha=0.5,
    cmap="viridis",
)
ax.axhline(1.0, color="black", lw=1.0, ls="--")
ax.set_xlabel("engine speed (rpm)")
ax.set_ylabel("AFR measured / AFR target")
ax.set_title("Fuelling error against engine speed")
fig.colorbar(scatter, label="MAP (kPa abs)")
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
ve_ratio = result.ve_true / result.ve_commanded
residual = np.abs(error_ratio[fuelled] - ve_ratio[fuelled])
print(f"max |AFR ratio - VE ratio| = {residual.max():.2e}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(result.time_s, result.pulse_width_ms, lw=1.0, color="tab:purple")
axes[0].set_ylabel("pulse width (ms)")

axes[1].plot(result.time_s, result.injector_duty * 100, lw=1.0, color="tab:brown")
axes[1].axhline(85, color="red", lw=1.0, ls="--", label="duty limit")
axes[1].set_ylabel("injector duty (%)")
axes[1].set_xlabel("time (s)")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("Commanded injector output")
fig.tight_layout()

In [ ]:
from vetuner.ve_surface import DEFAULT_MAP_AXIS, DEFAULT_RPM_AXIS

truth_table = ve_table(params=TRUE_SURFACE)
extent = [DEFAULT_RPM_AXIS[0], DEFAULT_RPM_AXIS[-1],
          DEFAULT_MAP_AXIS[0], DEFAULT_MAP_AXIS[-1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for ax, data, title, cmap in zip(
    axes,
    [truth_table * 100, base_table * 100, (base_table / truth_table - 1) * 100],
    ["True VE (%)", "ECU table VE (%)", "Table error (%)"],
    ["viridis", "viridis", "RdBu_r"],
    strict=True,
):
    limit = np.abs(data).max() if cmap == "RdBu_r" else None
    im = ax.imshow(
        data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap,
        vmin=-limit if limit else None, vmax=limit if limit else None,
    )
    ax.set_title(title)
    ax.set_xlabel("rpm")
    fig.colorbar(im, ax=ax)

axes[0].set_ylabel("MAP (kPa abs)")
fig.tight_layout()